<a href="https://colab.research.google.com/github/Rinosa123/Bilingual-Enterprise-RAG-Copilot/blob/main/notebooks/02_hybrid_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic–English Hybrid Retrieval Benchmark

This notebook evaluates three retrieval approaches:

1. BM25 keyword retrieval
2. Multilingual dense retrieval
3. Hybrid retrieval using Reciprocal Rank Fusion

The evaluation compares Top-1 accuracy, Hit@3 and Mean Reciprocal Rank.

In [1]:
%cd /content

!if [ -d "Bilingual-Enterprise-RAG-Copilot/.git" ]; then \
    git -C Bilingual-Enterprise-RAG-Copilot pull --ff-only origin main; \
else \
    git clone https://github.com/Rinosa123/Bilingual-Enterprise-RAG-Copilot.git; \
fi

%cd /content/Bilingual-Enterprise-RAG-Copilot

!pip -q install sentence-transformers

/content
Cloning into 'Bilingual-Enterprise-RAG-Copilot'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 56 (delta 16), reused 40 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 31.86 KiB | 1.10 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/Bilingual-Enterprise-RAG-Copilot


In [2]:
from pathlib import Path
import inspect

import numpy as np
import sentence_transformers
import torch
from sentence_transformers import SentenceTransformer

from src.ingestion.chunker import chunk_documents
from src.ingestion.text_loader import load_text_documents
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_retriever import (
    reciprocal_rank_fusion,
)


PROJECT_ROOT = Path.cwd()
DOCUMENT_DIRECTORY = PROJECT_ROOT / "data" / "sample_docs"

MODEL_NAME = "intfloat/multilingual-e5-small"


documents = load_text_documents(DOCUMENT_DIRECTORY)
chunks = chunk_documents(documents)

model = SentenceTransformer(MODEL_NAME)


print("Sentence Transformers:", sentence_transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Model:", MODEL_NAME)
print("Device:", model.device)
print("Documents:", len(documents))
print("Chunks:", len(chunks))

print(
    "BM25 constructor:",
    inspect.signature(BM25Retriever),
)

print(
    "BM25 search method:",
    inspect.signature(BM25Retriever.search),
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Sentence Transformers: 5.6.0
PyTorch: 2.11.0+cu128
CUDA available: True
Model: intfloat/multilingual-e5-small
Device: cuda:0
Documents: 2
Chunks: 10
BM25 constructor: (chunks: list[src.ingestion.chunker.TextChunk], k1: float = 1.5, b: float = 0.75) -> None
BM25 search method: (self, query: str, top_k: int = 3) -> list[src.retrieval.bm25_retriever.SearchResult]


## Build BM25 and Dense Retrieval Indexes

BM25 represents exact keyword matching, while multilingual E5 represents semantic meaning across English and Arabic.

In [3]:
# Build the BM25 keyword index.
bm25_retriever = BM25Retriever(chunks)


# Convert every chunk into the E5 passage format.
passage_texts = [
    f"passage: {chunk.section}\n{chunk.text}"
    for chunk in chunks
]


# Create normalized dense embeddings.
passage_embeddings = model.encode(
    passage_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)


# Create lookup tables for later evaluation.
chunk_by_id = {
    chunk.chunk_id: chunk
    for chunk in chunks
}

chunk_index_by_id = {
    chunk.chunk_id: index
    for index, chunk in enumerate(chunks)
}


print(
    "BM25 indexed chunks:",
    len(chunks),
)

print(
    "Dense embedding shape:",
    passage_embeddings.shape,
)


# Run a small BM25 verification query.
sample_results = bm25_retriever.search(
    "annual leave days",
    top_k=3,
)

print("\nSample BM25 ranking:")

for rank, result in enumerate(
    sample_results,
    start=1,
):
    print(
        f"{rank}. {result.chunk.chunk_id} | "
        f"score={result.score:.4f} | "
        f"section={result.chunk.section}"
    )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

BM25 indexed chunks: 10
Dense embedding shape: (10, 384)

Sample BM25 ranking:
1. HR-EN-001-CH-003 | score=8.3205 | section=2. Annual Leave
2. HR-EN-001-CH-004 | score=1.9751 | section=3. Remote Work


## Generate BM25, Dense and Hybrid Rankings

In [4]:
EVALUATION_QUERIES = (
    (
        "English question -> English document",
        "How many annual leave days do full-time employees receive?",
        "HR-EN-001-CH-003",
    ),
    (
        "Arabic question -> Arabic document",
        "ما الحد الأقصى لتكلفة الفندق؟",
        "HR-AR-001-CH-003",
    ),
    (
        "Arabic question -> English document",
        "كم عدد أيام الإجازة السنوية للموظف؟",
        "HR-EN-001-CH-003",
    ),
    (
        "English question -> Arabic document",
        "When must an expense claim be submitted?",
        "HR-AR-001-CH-002",
    ),
)


# Generate one dense embedding for every question.
query_texts = [
    f"query: {question}"
    for _, question, _ in EVALUATION_QUERIES
]

query_embeddings = model.encode(
    query_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
)


evaluation_rankings = []


for query_index, (
    label,
    question,
    expected_chunk_id,
) in enumerate(EVALUATION_QUERIES):

    # BM25 ranking
    bm25_results = bm25_retriever.search(
        question,
        top_k=len(chunks),
    )

    bm25_ranking = [
        result.chunk.chunk_id
        for result in bm25_results
    ]

    # Dense ranking
    dense_scores = (
        query_embeddings[query_index]
        @ passage_embeddings.T
    )

    dense_indices = np.argsort(-dense_scores)

    dense_ranking = [
        chunks[int(index)].chunk_id
        for index in dense_indices
    ]

    # Hybrid RRF ranking
    hybrid_results = reciprocal_rank_fusion(
        {
            "bm25": bm25_ranking,
            "dense": dense_ranking,
        }
    )

    hybrid_ranking = [
        result.chunk_id
        for result in hybrid_results
    ]

    evaluation_rankings.append(
        {
            "label": label,
            "question": question,
            "expected": expected_chunk_id,
            "bm25": bm25_ranking,
            "dense": dense_ranking,
            "hybrid": hybrid_ranking,
        }
    )

    print("=" * 80)
    print("Test:", label)
    print("Question:", question)
    print("Expected:", expected_chunk_id)

    print(
        "BM25 top 3:",
        bm25_ranking[:3],
    )

    print(
        "Dense top 3:",
        dense_ranking[:3],
    )

    print(
        "Hybrid top 3:",
        hybrid_ranking[:3],
    )

Test: English question -> English document
Question: How many annual leave days do full-time employees receive?
Expected: HR-EN-001-CH-003
BM25 top 3: ['HR-EN-001-CH-003', 'HR-EN-001-CH-004', 'HR-EN-001-CH-002']
Dense top 3: ['HR-EN-001-CH-003', 'HR-EN-001-CH-002', 'HR-EN-001-CH-004']
Hybrid top 3: ['HR-EN-001-CH-003', 'HR-EN-001-CH-004', 'HR-EN-001-CH-002']
Test: Arabic question -> Arabic document
Question: ما الحد الأقصى لتكلفة الفندق؟
Expected: HR-AR-001-CH-003
BM25 top 3: ['HR-AR-001-CH-003']
Dense top 3: ['HR-AR-001-CH-003', 'HR-AR-001-CH-002', 'HR-AR-001-CH-005']
Hybrid top 3: ['HR-AR-001-CH-003', 'HR-AR-001-CH-002', 'HR-AR-001-CH-005']
Test: Arabic question -> English document
Question: كم عدد أيام الإجازة السنوية للموظف؟
Expected: HR-EN-001-CH-003
BM25 top 3: ['HR-AR-001-CH-004', 'HR-AR-001-CH-005']
Dense top 3: ['HR-AR-001-CH-004', 'HR-AR-001-CH-003', 'HR-AR-001-CH-005']
Hybrid top 3: ['HR-AR-001-CH-004', 'HR-AR-001-CH-005', 'HR-AR-001-CH-003']
Test: English question -> Arabic

## Retrieval Evaluation Results

In [5]:
import pandas as pd


def find_expected_rank(
    ranking: list[str],
    expected_chunk_id: str,
) -> int | None:
    """Return the one-based rank of the expected chunk."""

    try:
        return ranking.index(expected_chunk_id) + 1
    except ValueError:
        return None


# Build a per-question rank table.
rank_rows = []

for evaluation in evaluation_rankings:
    rank_rows.append(
        {
            "Test": evaluation["label"],
            "BM25 Rank": find_expected_rank(
                evaluation["bm25"],
                evaluation["expected"],
            ),
            "Dense Rank": find_expected_rank(
                evaluation["dense"],
                evaluation["expected"],
            ),
            "Hybrid Rank": find_expected_rank(
                evaluation["hybrid"],
                evaluation["expected"],
            ),
        }
    )


rank_table = pd.DataFrame(rank_rows)

display(rank_table)


def calculate_metrics(
    method_name: str,
) -> dict[str, float | str]:
    """Calculate Top-1, Hit@3 and MRR."""

    ranks = [
        find_expected_rank(
            evaluation[method_name],
            evaluation["expected"],
        )
        for evaluation in evaluation_rankings
    ]

    query_count = len(ranks)

    top_1 = sum(
        rank == 1
        for rank in ranks
    ) / query_count

    hit_at_3 = sum(
        rank is not None and rank <= 3
        for rank in ranks
    ) / query_count

    mean_reciprocal_rank = sum(
        1 / rank
        if rank is not None
        else 0
        for rank in ranks
    ) / query_count

    return {
        "Retriever": method_name.upper(),
        "Top-1 Accuracy": top_1,
        "Hit@3": hit_at_3,
        "MRR": mean_reciprocal_rank,
    }


metrics_table = pd.DataFrame(
    [
        calculate_metrics("bm25"),
        calculate_metrics("dense"),
        calculate_metrics("hybrid"),
    ]
)


display(
    metrics_table.style.format(
        {
            "Top-1 Accuracy": "{:.0%}",
            "Hit@3": "{:.0%}",
            "MRR": "{:.4f}",
        }
    )
)


print("\nPlain-text summary:")

for _, row in metrics_table.iterrows():
    print(
        f"{row['Retriever']}: "
        f"Top-1={row['Top-1 Accuracy']:.0%}, "
        f"Hit@3={row['Hit@3']:.0%}, "
        f"MRR={row['MRR']:.4f}"
    )

,Test,BM25 Rank,Dense Rank,Hybrid Rank
0,English question -> English document,1.0,1,1
1,Arabic question -> Arabic document,1.0,1,1
2,Arabic question -> English document,NaN,4,4
3,English question -> Arabic document,NaN,1,5


,Retriever,Top-1 Accuracy,Hit@3,MRR
0,BM25,50%,50%,0.5000
1,DENSE,75%,75%,0.8125
2,HYBRID,50%,50%,0.6125



Plain-text summary:
BM25: Top-1=50%, Hit@3=50%, MRR=0.5000
DENSE: Top-1=75%, Hit@3=75%, MRR=0.8125
HYBRID: Top-1=50%, Hit@3=50%, MRR=0.6125


## Findings and Engineering Decision

- BM25 achieved 50% Top-1 accuracy and worked well for exact same-language keyword matching.
- Multilingual dense retrieval achieved the best result: 75% Top-1 accuracy and an MRR of 0.8125.
- Equal-weight Reciprocal Rank Fusion achieved 50% Top-1 accuracy.
- RRF promoted chunks supported by both BM25 and dense retrieval, but this reduced cross-language performance because BM25 cannot connect Arabic queries with English evidence, or English queries with Arabic evidence.

### Decision

Dense retrieval will be used as the primary multilingual candidate retriever. BM25 will remain available for exact keyword and identifier matching. The next pipeline stage will apply a multilingual cross-encoder reranker to the combined candidate set.

### Limitation

This is an initial demonstration using two synthetic policy documents and four evaluation questions. A larger bilingual evaluation dataset will be added before drawing production-level conclusions.